# Clase 029 — pivot_table y crosstab

**Parte 0** · VanderPlas cap. 3 § 3.10.

> 🎯 Resumen rápido tipo Excel + tablas de contingencia.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

df = pd.DataFrame({
    'species': np.repeat(['Adelie', 'Chinstrap', 'Gentoo'], [12, 8, 10]),
    'island' : rng.choice(['Biscoe', 'Dream', 'Torgersen'], 30),
    'sex'    : rng.choice(['M', 'F'], 30),
    'masa'   : rng.normal(4200, 600, 30),
})
print(df.head())

## 🧠 Intuición previa

Una **pivot_table** reorganiza filas sueltas en una **grilla de doble entrada** (como una tabla dinámica de Excel): elegís qué variable va en las **filas**, cuál en las **columnas**, y qué número resumido (media, suma, conteo) llena cada **celda**. Cuando el índice es jerárquico, pensalo como **carpetas anidadas** (`especie > sexo`): cada combinación cae en su propio casillero.

## 1️⃣ `pivot_table` — el Excel de pandas

```python
df.pivot_table(
    index='species',
    columns='sex',
    values='masa',
    aggfunc='mean',
    margins=True,    # totales
)
```

In [ ]:
pivot = df.pivot_table(
    index='species',
    columns='sex',
    values='masa',
    aggfunc='mean',
)
print('mean masa por species × sex:')
print(pivot.round(0))

## 2️⃣ Con totales (`margins=True`)

In [ ]:
pivot_m = df.pivot_table(
    index='species', columns='sex', values='masa',
    aggfunc='mean', margins=True, margins_name='Total',
)
print(pivot_m.round(0))

## 3️⃣ Múltiples aggfunc

In [ ]:
pivot_multi = df.pivot_table(
    index='species', columns='sex', values='masa',
    aggfunc=['mean', 'count'],
)
print(pivot_multi.round(0))

## 4️⃣ `crosstab` — contingencia

Count cuando se cruzan dos categóricas:

In [ ]:
ct = pd.crosstab(df['species'], df['island'])
print('counts species × island:')
print(ct)
print('\nnormalizado por fila (% por species):')
print(pd.crosstab(df['species'], df['island'], normalize='index').round(2))
print('\nnormalizado total:')
print(pd.crosstab(df['species'], df['island'], normalize='all').round(3))

## 5️⃣ `pivot` (sin agregar)

`pivot` falla si hay duplicados en la combinación `(index, columns)`. **Solo úsalo cuando garantizas unicidad** — para todo lo demás, `pivot_table`.

## 6️⃣ Heatmap rápido

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
im = ax.imshow(pivot.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('sex')
ax.set_title('Mean masa (g)')
plt.colorbar(im)

# anotar valores
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', color='white')
plt.tight_layout()
plt.show()

## ✅ Checklist

- [ ] Sé construir pivot_table con index/columns/values/aggfunc
- [ ] Añado margins=True para totales
- [ ] Uso crosstab para counts entre categóricas
- [ ] Normalizo crosstab por fila/col/total
- [ ] Sé que `pivot` (sin _table) requiere unicidad

## 📝 Homework

Ver `README.md`. pivot species × island, crosstab + normalize, heatmap.

## 📖 Definiciones y características

**`pivot_table`**

Resumen tabular estilo Excel: defines `index`, `columns`, `values` y `aggfunc`. Acepta duplicados (agrega). El atajo más usado para reportes.

**`pivot` (sin _table)**

Variante que NO agrega — falla si hay duplicados en (index, columns). Más estricta; úsala solo cuando garantizas unicidad.

**`crosstab`**

Tabla de contingencia entre 2 categóricas: counts cruzados. Con `normalize='index'`/`'columns'`/`'all'` muestra proporciones.

**`margins=True`**

Añade fila/columna "Total" al pivot. Útil para verificar manualmente y para reportes ejecutivos.

**Heatmap de pivot**

Renderizar el pivot como matriz coloreada (`plt.imshow` o `seaborn.heatmap`) — patrones visuales saltan a la vista.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `pivot()` lanza `ValueError: Index contains duplicate entries` | Hay duplicados en (index, columns). **Fix**: usa `pivot_table()` con `aggfunc='sum'`/'mean' que agrega los duplicados, o agrega antes con groupby. |
| `pivot_table` da NaN donde no hay datos | Combinaciones (index × columns) sin filas. **Fix**: `fill_value=0` (o el default que tenga sentido). |
| `crosstab` cuenta cosas raras con muchos NaN | Crosstab cuenta filas no-NaN por default. **Fix**: filtra previamente o pasa `dropna=False`. |
| Pivot con cols numéricas float queda feo | Sin `aggfunc` explícito, pandas usa mean. Si querías sum, espécifica: `aggfunc='sum'`. |
| Plot del pivot rompe por MultiIndex | Pivot con múltiples niveles de columnas → MultiIndex. **Fix**: aplana con `pivot.columns = ['_'.join(c) for c in pivot.columns]` antes del plot. |

## ❓ Preguntas frecuentes

**❓ ¿`pivot_table` o `groupby` + `unstack`?**

Equivalentes en resultado. **`pivot_table`** es más declarativo, mejor para reportes. **`groupby + unstack`** más componible, mejor en pipelines.

**❓ ¿`crosstab` o `pivot_table` con `aggfunc='count'`?**

Equivalentes para counts. **`crosstab`** tiene API más simple para 2 categóricas. **`pivot_table`** más flexible (varias values, varias funcs).

**❓ ¿Cómo ordeno el pivot?**

Por valores: `pivot.sort_values('col_x', ascending=False)`. Por suma: `pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]`.

**❓ ¿Reportes Excel-like exportables?**

`pivot.to_excel('reporte.xlsx')` directo. O `to_csv` para CSV. Para formato fino (colores, formulas), usa `openpyxl` o `xlsxwriter`.

**❓ ¿Cuándo no usar pivot?**

Cuando los datos ya están en formato wide y solo necesitas plot/agregaciones — usar `groupby` directo. Pivot es para transformar long → wide.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.10
- [pandas Reshaping](https://pandas.pydata.org/docs/user_guide/reshaping.html)

➡️ **Siguiente:** [030 — Operaciones sobre strings](../030-pandas-operaciones-vectorizadas-sobre-strings/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Pivot básico** (species x sex, media de masa).

In [ ]:
import numpy as np, pandas as pd

def make_penguins(seed=42, with_na=False):
    """DataFrame sintetico estilo Palmer Penguins (344 filas), sin internet."""
    rng = np.random.default_rng(seed)
    cfg = {  # especie: (n, islas, bill_len, bill_depth, flipper, body_mass)
        'Adelie':    (152, ['Torgersen', 'Biscoe', 'Dream'], 38.8, 18.3, 190, 3700),
        'Chinstrap': (68,  ['Dream'],                        48.8, 18.4, 196, 3733),
        'Gentoo':    (124, ['Biscoe'],                       47.5, 15.0, 217, 5076),
    }
    filas = []
    for sp, (n, islas, bl, bd, fl, bm) in cfg.items():
        for _ in range(n):
            sex = rng.choice(['male', 'female'])
            k = 1.0 if sex == 'male' else 0.93
            filas.append({
                'species': sp,
                'island': rng.choice(islas),
                'bill_length_mm': round(float(rng.normal(bl, 2.5)), 1),
                'bill_depth_mm': round(float(rng.normal(bd, 1.2)), 1),
                'flipper_length_mm': float(round(rng.normal(fl, 6))),
                'body_mass_g': float(round(rng.normal(bm * k, 300))),
                'sex': sex,
            })
    df = pd.DataFrame(filas)
    if with_na:
        idx = rng.choice(df.index, size=12, replace=False)
        df.loc[idx[:6], 'bill_length_mm'] = np.nan
        df.loc[idx[6:], 'sex'] = np.nan
    return df

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
df = make_penguins()
piv = df.pivot_table(index='species', columns='sex', values='body_mass_g', aggfunc='mean')
print(piv.round(1))
assert piv.shape[0] == 3

**Ej. 2 — Pivot con totales** (`margins=True`).

In [ ]:
piv_m = df.pivot_table(index='species', columns='sex', values='body_mass_g',
                       aggfunc='mean', margins=True, margins_name='TOTAL')
print(piv_m.round(1))
assert 'TOTAL' in piv_m.index

**Ej. 3 — Crosstab de conteos** (species x island).

In [ ]:
ct = pd.crosstab(df['species'], df['island'])
print(ct)
assert ct.values.sum() == 344

**Ej. 4 — Crosstab normalizado por fila** (`normalize='index'`).

In [ ]:
ctn = pd.crosstab(df['species'], df['island'], normalize='index')
print((ctn * 100).round(1))
assert np.allclose(ctn.sum(axis=1), 1.0)   # cada fila suma 100%

**Ej. 5 — Pivot -> heatmap** con `imshow`.

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(piv.values, cmap='viridis')
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels(piv.index)
fig.colorbar(im, label='body_mass_g medio')
plt.close(fig)          # no lo mostramos, solo verificamos que se construye
assert im is not None
print('Heatmap del pivot construido correctamente.')